# Observational Comparisons with Galactic Wind Models

This notebook focuses on generating observables from wind models for direct comparison with observational data.

Key observables:
- Velocity distributions (dN/dv)
- Column density distributions
- Velocity moments (mean, dispersion)
- Mass outflow rates

In [ ]:
import sys
sys.path.insert(0, '..')

from multiphasegalacticwind import WindModel, WindConfig
from multiphasegalacticwind import (
    plot_velocity_distribution,
    plot_column_density_distribution
)
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 14

## 1. Starburst Galaxy Model

Create a model representative of local starburst galaxies like M82.

In [ ]:
# M82-like starburst parameters
model_m82 = WindModel(
    # Galaxy properties
    v_circ=130.0,          # km/s, from rotation curve
    
    # Starburst properties
    SFR=10.0,              # Msun/yr
    eta_M=0.3,             # Hot mass loading
    eta_M_cold=3.0,        # Cold mass loading (observed)
    
    # Wind launch
    r_star_kpc=0.2,        # Small starburst region
    
    # Cloud properties
    N_cloud_species=10,
    cloud_alpha=1.8,       # Shallower than Salpeter
    cloud_mass_range=(10, 1e4),  # Msun
    
    # Solver
    r_max_kpc=20.0,
    rtol=1e-6
)

print("Running M82-like model...")
sol_m82 = model_m82.run()
print(f"Integration reached {sol_m82.r[-1]:.1f} kpc")

## 2. Velocity Distribution Analysis

Calculate the velocity distribution that would be observed in absorption or emission.

In [ ]:
# Calculate velocity distribution
v_cloud, dN_dv = sol_m82.calculate_velocity_distribution()

# Create figure with multiple panels
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Linear scale velocity distribution
axes[0,0].plot(v_cloud, dN_dv, linewidth=2)
axes[0,0].set_xlabel('Velocity [km/s]')
axes[0,0].set_ylabel('dN/dv')
axes[0,0].set_title('Velocity Distribution')
axes[0,0].set_xlim(0, 800)
axes[0,0].grid(True, alpha=0.3)

# Panel 2: Log scale to see tail
axes[0,1].semilogy(v_cloud, dN_dv, linewidth=2)
axes[0,1].set_xlabel('Velocity [km/s]')
axes[0,1].set_ylabel('dN/dv')
axes[0,1].set_title('Velocity Distribution (Log Scale)')
axes[0,1].set_xlim(0, 1500)
axes[0,1].set_ylim(bottom=1e-6*np.max(dN_dv))
axes[0,1].grid(True, alpha=0.3)

# Panel 3: Cumulative distribution
cumulative = np.zeros_like(v_cloud)
for i in range(1, len(v_cloud)):
    cumulative[i] = np.trapz(dN_dv[:i], v_cloud[:i])
cumulative = cumulative / cumulative[-1]  # Normalize

axes[1,0].plot(v_cloud, cumulative, linewidth=2)
axes[1,0].set_xlabel('Velocity [km/s]')
axes[1,0].set_ylabel('Cumulative Fraction')
axes[1,0].set_title('Cumulative Velocity Distribution')
axes[1,0].set_xlim(0, 1000)
axes[1,0].grid(True, alpha=0.3)

# Add percentile markers
percentiles = [0.1, 0.5, 0.9]
for p in percentiles:
    v_p = np.interp(p, cumulative, v_cloud)
    axes[1,0].axhline(p, color='red', linestyle='--', alpha=0.5)
    axes[1,0].axvline(v_p, color='red', linestyle='--', alpha=0.5)
    axes[1,0].text(v_p+20, 0.05, f'{int(v_p)} km/s', rotation=90)

# Panel 4: Velocity moments
moments = sol_m82.calculate_velocity_moments()
if 'mean' in moments:
    info_text = f"Mean velocity: {moments['mean']:.0f} km/s\n"
    info_text += f"Velocity dispersion: {moments['dispersion']:.0f} km/s\n"
    if 'skewness' in moments:
        info_text += f"Skewness: {moments['skewness']:.2f}\n"
    
    # Percentile velocities
    v10 = np.interp(0.1, cumulative, v_cloud)
    v50 = np.interp(0.5, cumulative, v_cloud)
    v90 = np.interp(0.9, cumulative, v_cloud)
    info_text += f"\n10th percentile: {v10:.0f} km/s"
    info_text += f"\n50th percentile: {v50:.0f} km/s"
    info_text += f"\n90th percentile: {v90:.0f} km/s"
    info_text += f"\nv90/v10 ratio: {v90/v10:.1f}"
else:
    info_text = "Velocity moments not available\n(integration may have terminated early)"

axes[1,1].text(0.1, 0.5, info_text, transform=axes[1,1].transAxes, 
               fontsize=14, verticalalignment='center',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1,1].axis('off')
axes[1,1].set_title('Velocity Statistics')

plt.suptitle('M82-like Starburst: Velocity Distribution Analysis', fontsize=16)
plt.tight_layout()

## 3. Column Density Distribution

For absorption line studies (e.g., COS-Halos, COS-Burst), we need column density per velocity bin.

In [ ]:
# Calculate column density distribution
v_cloud, dN_dv_dict = sol_m82.calculate_column_density_by_species()

# Create figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# Top panel: Column density by species
colors = plt.cm.viridis(np.linspace(0, 1, len(dN_dv_dict['species'])))

for i, (dN_dv_i, M_i) in enumerate(zip(dN_dv_dict['species'], dN_dv_dict['M_cloud0'])):
    ax1.semilogy(v_cloud, dN_dv_i, color=colors[i], 
                 label=f'M = {M_i/1.989e33:.0f} M$_\odot$', alpha=0.7)

ax1.semilogy(v_cloud, dN_dv_dict['total'], 'k-', linewidth=3, label='Total')
ax1.set_ylabel('dN/dv [cm$^{-2}$ / (km/s)]')
ax1.set_title('Column Density Distribution by Cloud Mass')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1000)
ax1.set_ylim(bottom=1e8)

# Bottom panel: Integrated column density
# Calculate N(>v)
N_gt_v = np.zeros_like(v_cloud)
for i in range(len(v_cloud)):
    N_gt_v[i] = np.trapz(dN_dv_dict['total'][i:], v_cloud[i:])

ax2.loglog(v_cloud[v_cloud>10], N_gt_v[v_cloud>10], linewidth=2)
ax2.set_xlabel('Velocity [km/s]')
ax2.set_ylabel('N(>v) [cm$^{-2}$]')
ax2.set_title('Integrated Column Density above Velocity')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(10, 1000)

# Add reference lines for typical absorption thresholds
ax2.axhline(1e13, color='red', linestyle='--', alpha=0.5, label='CIV detection limit')
ax2.axhline(1e15, color='blue', linestyle='--', alpha=0.5, label='OVI detection limit')
ax2.legend()

plt.suptitle('M82-like Starburst: Column Density Analysis', fontsize=16)
plt.tight_layout()

## 4. Parameter Study: Effect of Cold Mass Loading

Observationally, cold mass loading is a key unknown. Let's see how it affects observables.

In [ ]:
# Range of cold mass loading factors
eta_M_cold_values = [0.5, 1.0, 2.0, 4.0]
solutions = []
colors = plt.cm.plasma(np.linspace(0, 1, len(eta_M_cold_values)))

# Run models
for eta_M_cold in eta_M_cold_values:
    print(f"Running model with eta_M_cold = {eta_M_cold}...")
    model = WindModel(
        SFR=10.0,
        v_circ=130.0,
        eta_M=0.3,
        eta_M_cold=eta_M_cold,
        N_cloud_species=8,
        r_max_kpc=20.0,
        rtol=1e-6
    )
    solutions.append(model.run())

# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Velocity distributions
for i, (sol, eta) in enumerate(zip(solutions, eta_M_cold_values)):
    v_cloud, dN_dv = sol.calculate_velocity_distribution()
    axes[0,0].plot(v_cloud, dN_dv/np.max(dN_dv), color=colors[i], 
                   label=f'$\eta_{{M,cold}} = {eta}$', linewidth=2)

axes[0,0].set_xlabel('Velocity [km/s]')
axes[0,0].set_ylabel('Normalized dN/dv')
axes[0,0].set_title('Velocity Distribution vs Cold Mass Loading')
axes[0,0].set_xlim(0, 800)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Column density distributions
for i, (sol, eta) in enumerate(zip(solutions, eta_M_cold_values)):
    v_cloud, dN_dv_col = sol.calculate_column_density_distribution()
    axes[0,1].semilogy(v_cloud, dN_dv_col, color=colors[i], 
                       label=f'$\eta_{{M,cold}} = {eta}$', linewidth=2)

axes[0,1].set_xlabel('Velocity [km/s]')
axes[0,1].set_ylabel('dN/dv [cm$^{-2}$ / (km/s)]')
axes[0,1].set_title('Column Density vs Cold Mass Loading')
axes[0,1].set_xlim(0, 800)
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Mean velocity vs radius
for i, (sol, eta) in enumerate(zip(solutions, eta_M_cold_values)):
    # Calculate mean cloud velocity at each radius
    mean_v_cloud = np.mean(sol.v_cl, axis=0)
    axes[1,0].semilogx(sol.r, mean_v_cloud, color=colors[i], 
                       label=f'$\eta_{{M,cold}} = {eta}$', linewidth=2)

axes[1,0].set_xlabel('Radius [kpc]')
axes[1,0].set_ylabel('Mean Cloud Velocity [km/s]')
axes[1,0].set_title('Cloud Velocity Evolution')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Velocity dispersion
dispersions = []
for i, (sol, eta) in enumerate(zip(solutions, eta_M_cold_values)):
    moments = sol.calculate_velocity_moments()
    if 'dispersion' in moments:
        dispersions.append(moments['dispersion'])
    else:
        dispersions.append(np.nan)

axes[1,1].plot(eta_M_cold_values, dispersions, 'o-', markersize=10, linewidth=2)
axes[1,1].set_xlabel('$\eta_{M,cold}$')
axes[1,1].set_ylabel('Velocity Dispersion [km/s]')
axes[1,1].set_title('Velocity Dispersion vs Cold Mass Loading')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Effect of Cold Mass Loading on Observables', fontsize=16)
plt.tight_layout()

## 5. Mock Observations: Line Profiles

Generate mock absorption line profiles for comparison with UV spectroscopy.

In [ ]:
# Use the eta_M_cold = 2.0 model
sol = solutions[2]  # eta_M_cold = 2.0

# Get column density distribution
v_cloud, dN_dv_col = sol.calculate_column_density_distribution()

# Create mock absorption profile
# Assume Doppler parameter b = 20 km/s
b_doppler = 20.0  # km/s

# Create fine velocity grid
v_fine = np.linspace(-200, 1000, 2000)
tau = np.zeros_like(v_fine)

# For each cloud velocity, add Gaussian component
for i in range(len(v_cloud)-1):
    if dN_dv_col[i] > 0:
        # Column density in this velocity bin
        N_i = dN_dv_col[i] * (v_cloud[i+1] - v_cloud[i])
        
        # Add Gaussian profile centered at v_cloud[i]
        profile = np.exp(-(v_fine - v_cloud[i])**2 / (2 * b_doppler**2))
        profile = profile / (b_doppler * np.sqrt(2*np.pi))
        
        # Optical depth (arbitrary normalization for illustration)
        tau += N_i * profile * 1e-14  # Scale factor for visualization

# Convert to normalized flux
flux = np.exp(-tau)

# Create figure with multiple ions
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

# Different ions have different covering fractions and populations
ion_properties = [
    {'name': 'CII', 'covering': 1.0, 'strength': 1.0},
    {'name': 'CIV', 'covering': 0.7, 'strength': 0.5},
    {'name': 'OVI', 'covering': 0.5, 'strength': 0.3}
]

for ax, ion in zip(axes, ion_properties):
    # Scale optical depth by ion properties
    tau_ion = tau * ion['strength']
    flux_ion = ion['covering'] * np.exp(-tau_ion) + (1 - ion['covering'])
    
    ax.plot(v_fine, flux_ion, 'b-', linewidth=1)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(0.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel(f'{ion["name"]} Normalized Flux')
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.1, f'Covering = {ion["covering"]}', 
            transform=ax.transAxes, fontsize=12)

axes[-1].set_xlabel('Velocity [km/s]')
axes[-1].set_xlim(-100, 800)

plt.suptitle('Mock UV Absorption Line Profiles', fontsize=16)
plt.tight_layout()

# Print equivalent widths
print("Approximate equivalent widths:")
for ion in ion_properties:
    tau_ion = tau * ion['strength']
    flux_ion = ion['covering'] * np.exp(-tau_ion) + (1 - ion['covering'])
    EW = np.trapz(1 - flux_ion, v_fine)  # km/s
    print(f"  {ion['name']}: {EW:.0f} km/s")

## 6. Observable Diagnostics Summary

Create a summary plot of key observables for the model suite.

In [ ]:
# Compile diagnostics for all models
diagnostics = {
    'eta_M_cold': eta_M_cold_values,
    'v_mean': [],
    'v_disp': [],
    'v_90': [],
    'N_total': [],
    'covering_300': []  # Fraction of gas with v > 300 km/s
}

for sol in solutions:
    # Velocity moments
    moments = sol.calculate_velocity_moments()
    if 'mean' in moments:
        diagnostics['v_mean'].append(moments['mean'])
        diagnostics['v_disp'].append(moments['dispersion'])
    else:
        diagnostics['v_mean'].append(np.nan)
        diagnostics['v_disp'].append(np.nan)
    
    # 90th percentile velocity
    v_cloud, dN_dv = sol.calculate_velocity_distribution()
    cumulative = np.cumsum(dN_dv)
    cumulative = cumulative / cumulative[-1]
    v_90 = np.interp(0.9, cumulative, v_cloud)
    diagnostics['v_90'].append(v_90)
    
    # Total column density
    v_cloud, dN_dv_col = sol.calculate_column_density_distribution()
    N_total = np.trapz(dN_dv_col, v_cloud)
    diagnostics['N_total'].append(N_total)
    
    # High velocity covering
    frac_300 = np.interp(300, v_cloud, cumulative)
    diagnostics['covering_300'].append(1 - frac_300)

# Create summary figure
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Plot diagnostics
axes[0].plot(diagnostics['eta_M_cold'], diagnostics['v_mean'], 'o-', markersize=8)
axes[0].set(xlabel='$\eta_{M,cold}$', ylabel='Mean Velocity [km/s]', 
            title='Mean Outflow Velocity')

axes[1].plot(diagnostics['eta_M_cold'], diagnostics['v_disp'], 'o-', markersize=8)
axes[1].set(xlabel='$\eta_{M,cold}$', ylabel='Velocity Dispersion [km/s]',
            title='Velocity Dispersion')

axes[2].plot(diagnostics['eta_M_cold'], diagnostics['v_90'], 'o-', markersize=8)
axes[2].set(xlabel='$\eta_{M,cold}$', ylabel='$v_{90}$ [km/s]',
            title='90th Percentile Velocity')

axes[3].semilogy(diagnostics['eta_M_cold'], diagnostics['N_total'], 'o-', markersize=8)
axes[3].set(xlabel='$\eta_{M,cold}$', ylabel='$N_{total}$ [cm$^{-2}$]',
            title='Total Column Density')

axes[4].plot(diagnostics['eta_M_cold'], diagnostics['covering_300'], 'o-', markersize=8)
axes[4].set(xlabel='$\eta_{M,cold}$', ylabel='Fraction with v > 300 km/s',
            title='High Velocity Covering Fraction')

# Hide last subplot
axes[5].axis('off')

# Add text summary
summary_text = "Observable Trends:\n\n"
summary_text += "• Higher cold mass loading →\n"
summary_text += "  - Lower mean velocities\n"
summary_text += "  - Higher velocity dispersion\n"
summary_text += "  - More total column density\n"
summary_text += "  - Lower high-v covering\n\n"
summary_text += "Key for observations:\n"
summary_text += "• Velocity moments constrain\n"
summary_text += "  mass loading & mixing"

axes[5].text(0.1, 0.5, summary_text, transform=axes[5].transAxes,
             fontsize=12, verticalalignment='center')

for ax in axes[:-1]:
    ax.grid(True, alpha=0.3)

plt.suptitle('Observable Diagnostics vs Cold Mass Loading', fontsize=16)
plt.tight_layout()

# Print summary table
print("\nSummary Table:")
print(f"{'eta_M_cold':>10} {'v_mean':>10} {'v_disp':>10} {'v_90':>10} {'N_total':>12}")
print("-" * 55)
for i in range(len(eta_M_cold_values)):
    print(f"{diagnostics['eta_M_cold'][i]:>10.1f} "
          f"{diagnostics['v_mean'][i]:>10.0f} "
          f"{diagnostics['v_disp'][i]:>10.0f} "
          f"{diagnostics['v_90'][i]:>10.0f} "
          f"{diagnostics['N_total'][i]:>12.2e}")

## Summary

This notebook demonstrated how to:

1. **Create realistic galaxy models** based on observed properties
2. **Calculate velocity distributions** for comparison with spectroscopy
3. **Compute column density distributions** for absorption line studies
4. **Generate mock absorption profiles** for direct comparison with UV spectra
5. **Study parameter dependencies** to understand observable trends

Key insights for observers:
- Cold mass loading strongly affects velocity distribution shape
- Higher mass loading → lower mean velocity but higher dispersion
- Column density scales with cold mass loading
- Multiple ions probe different phases and radii